## If working in Google Colab, run this first
Run the cell below at the start of every session. It downloads the course files to your Google Drive and keeps them up to date.
When you see "Setup complete", you are ready to run the rest of the notebook.

**Do not edit or save your work inside the `AIforMovementScience` folder.**
Use **File > Save a copy in Drive** and work on your own copy.

In [ ]:
# [course-setup] Run this cell first, every session (or use Runtime > Run all)
import os, sys
try:
    from google.colab import drive
    os.chdir('/content')                  # start from a valid directory
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/AIforMovementScience'
    if os.path.isdir(REPO + '/.git'):
        # already downloaded before: just update it
        !git -C "$REPO" fetch origin main && git -C "$REPO" reset --hard origin/main
    else:
        # first time, or a broken/partial copy: download a fresh copy
        !rm -rf "$REPO"
        !git clone https://github.com/dguari1/AIforMovementScience.git "$REPO"
except ImportError:
    # not in Colab: walk up from here until the repository root is found
    REPO = os.getcwd()
    while REPO != '/' and not os.path.isdir(os.path.join(REPO, 'lectures', 'data')):
        REPO = os.path.dirname(REPO)

WORKDIR = REPO + '/lectures'
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print('Setup complete. You can now run the rest of the notebook.')

# Small Datasets, Part 2: How Much Can We Trust the Result?

In Part 1, we computed gait features from the force insoles. In this notebook, we use the features to classify **controls vs. persons with Parkinson's disease (PD)**. We use logistic regression, because it is easy to interpret.

We will:
1. Train and tune a model on the small dataset (64 subjects).
2. Repeat the train/test split many times, and see how much the result changes.
3. Use a larger dataset (164 subjects, 3 studies) with **group-aware** splits, and see the difference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import (train_test_split, GridSearchCV, StratifiedShuffleSplit,
                                     RepeatedStratifiedKFold, cross_val_score,
                                     GroupShuffleSplit, GroupKFold, StratifiedKFold)
from sklearn.metrics import (accuracy_score, classification_report, roc_curve, roc_auc_score,
                             ConfusionMatrixDisplay)

## 1. The small dataset: 64 subjects

In [ ]:
data = pd.read_csv('data/gait_measures_PD_controls.csv')
print("Subjects:", data['ID'].nunique(), "| rows:", len(data))
print(data['class'].value_counts().rename({0: 'control', 1: 'PD'}))
data.head()

In [ ]:
X = data.drop(columns=['ID', 'class'])
y = data['class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=16, stratify=y)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x=y_train, hue=y_train, palette="Set2", ax=ax[0], legend=False)
ax[0].set_title("Class distribution in the training set")
sns.countplot(x=y_test, hue=y_test, palette="Set2", ax=ax[1], legend=False)
ax[1].set_title("Class distribution in the testing set")
plt.tight_layout()
plt.show()
print("Test subjects:", len(y_test))

The test set has only 13 subjects. One subject is 7.7% of the test accuracy.

### A first model

The features have different units, so we scale them in a `Pipeline`. We also use `class_weight='balanced'`, because the two classes do not have the same size.

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000))
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)                 # classes: for the report and the confusion matrix
y_prob = pipeline.predict_proba(X_test)[:, 1]     # probabilities: for the ROC curve

print(classification_report(y_test, y_pred, target_names=['control', 'PD']))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['control', 'PD'], cmap='Blues', ax=ax[0], colorbar=False)
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax[1].plot(fpr, tpr, marker='.', label=f'Logistic regression (AUC = {roc_auc_score(y_test, y_prob):.2f})')
ax[1].plot([0, 1], [0, 1], 'k--')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC curve (test set)')
ax[1].legend()
plt.tight_layout()
plt.show()

This is a reasonable starting point: the model classifies 10 of the 13 test subjects correctly. Note that the ROC curve uses the **probabilities** (`predict_proba`). With the predicted classes (`predict`), the ROC curve would have only one point between (0, 0) and (1, 1).

Which features does the model use? The features are scaled, so we can compare the sizes of the coefficients.

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns,
                        'Coefficient': pipeline.named_steps['model'].coef_[0]}).sort_values('Coefficient')
plt.figure(figsize=(8, 6))
plt.barh(coef_df['Feature'], coef_df['Coefficient'],
         color=['tab:red' if v < 0 else 'tab:blue' for v in coef_df['Coefficient']])
plt.axvline(0, color='k', lw=1)
plt.title('Logistic regression coefficients (scaled features)')
plt.xlabel('Coefficient value (positive = more probably PD)')
plt.tight_layout()
plt.show()

Some features have coefficients close to zero, so they do not help much. We can try to keep only the most useful features.

### Tune the model: feature selection and regularization

`SelectKBest` keeps the `k` features that best separate the classes (here with the ANOVA F-score, `f_classif`). We put it **inside** the pipeline, so that it selects the features with the training data of each cross-validation fold only. `GridSearchCV` tries every combination of `k` and the regularization strength `C` with 5-fold cross-validation on the training set.

In [ ]:
tune_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('select', SelectKBest(score_func=f_classif)),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000))
])
param_grid = {
    'select__k': [3, 5, 10, 'all'],
    'model__C': [0.01, 0.1, 1, 10, 100],
}
grid_search = GridSearchCV(tune_pipeline, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)
print(f"Cross-validation AUC (training set): {grid_search.best_score_:.2f}")
print(f"Test AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1]):.2f}")
print(f"Test accuracy: {accuracy_score(y_test, best_model.predict(X_test)):.2f}")
print("Selected features:", list(X.columns[best_model.named_steps['select'].get_support()]))

The tuned model uses only 5 features. Its test AUC is similar to the first model, but its test **accuracy is lower** (0.62 vs 0.77). A strong regularization (`C=0.01`) pushes all the probabilities close to 0.5, so many subjects fall on the wrong side of the 0.5 threshold, even when the ranking (the AUC) is good.

So which model is better? With 13 test subjects, we cannot really say. The next section shows why.

## 2. How much does one test result tell us?

We train the same first model with four different random splits. Only `random_state` changes.

In [ ]:
for seed in [0, 1, 16, 42]:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=seed, stratify=y)
    pipeline.fit(Xtr, ytr)
    auc = roc_auc_score(yte, pipeline.predict_proba(Xte)[:, 1])
    acc = accuracy_score(yte, pipeline.predict(Xte))
    print(f"random_state = {seed:2d}: test accuracy = {acc:.2f}, test AUC = {auc:.2f}")

The same model, with the same data, gives a test AUC from about 0.5 (a random guess) to 1.0 (perfect). Only the choice of the 13 test subjects changed!

Let us repeat the split 200 times and look at all the results.

In [ ]:
splitter = StratifiedShuffleSplit(n_splits=200, test_size=0.20, random_state=0)
aucs_small = cross_val_score(pipeline, X, y, cv=splitter, scoring='roc_auc')

plt.figure(figsize=(7, 4))
plt.hist(aucs_small, bins=20, color='tab:orange', edgecolor='k')
plt.axvline(aucs_small.mean(), color='k', ls='--', label=f'mean = {aucs_small.mean():.2f}')
plt.xlabel('Test AUC')
plt.ylabel('Number of splits')
plt.title('64 subjects: test AUC of 200 random splits')
plt.legend()
plt.show()

print(f"Mean AUC: {aucs_small.mean():.2f}, standard deviation: {aucs_small.std():.2f}")
print(f"Lowest: {aucs_small.min():.2f}, highest: {aucs_small.max():.2f}")
print(f"90% of the splits are between {np.percentile(aucs_small, 5):.2f} and {np.percentile(aucs_small, 95):.2f}")

**This is the problem with small datasets.** One train/test split can make the model look excellent or useless. If you report only one split, you report luck.

What can we do?
- **Report the mean and the spread** over many splits, for example with repeated k-fold cross-validation (below). Do not report the best split.
- **Do not trust small differences** between models. A difference of 0.05 in AUC is much smaller than the spread from one split to the next.
- **Get more data.** The result of a small dataset is a proof of principle, not a final model.

In [ ]:
# repeated stratified 5-fold cross-validation: 5 folds x 20 repeats = 100 test results
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=0)
scores = cross_val_score(pipeline, X, y, cv=rkf, scoring='roc_auc')
print(f"Repeated 5-fold CV: AUC = {scores.mean():.2f} ± {scores.std():.2f}")

## 3. More subjects: the combined dataset

The file `gait_measures_PD_controls_ALL.csv` has the same features for **three studies** of the PhysioNet database. The ID shows the study: `Ga`, `Ju` and `Si` (the study we used above).

In [ ]:
data_all = pd.read_csv('data/gait_measures_PD_controls_ALL.csv').dropna()
data_all['study'] = data_all['ID'].str[:2]

print("Rows:", len(data_all), "| subjects:", data_all['ID'].nunique())
data_all.groupby(['study', 'class']).agg(rows=('ID', 'size'), subjects=('ID', 'nunique'))

Two important differences from the small dataset:

1. **Many subjects have more than one row** (more than one recording). So we must split by **subject**, not by row (notebook 04.03). We use `GroupShuffleSplit` and `GroupKFold` with the `ID` column.
2. **The three studies used different protocols** (for example, walking with and without a second task). The studies also have different numbers of controls and patients. A model could learn some differences between the studies, not only the differences between PD and controls. Keep this in mind when you read the results.

We repeat the 200 random splits, now **by subject**, with 20% of the subjects in each test set.

In [ ]:
X_all = data_all.drop(columns=['ID', 'class', 'study'])
y_all = data_all['class']
groups_all = data_all['ID']

group_splitter = GroupShuffleSplit(n_splits=200, test_size=0.20, random_state=0)
aucs_all = cross_val_score(pipeline, X_all, y_all, cv=group_splitter, groups=groups_all, scoring='roc_auc')

plt.figure(figsize=(7, 4))
bins = np.linspace(0.2, 1.0, 33)
plt.hist(aucs_small, bins=bins, alpha=0.6, color='tab:orange', edgecolor='k', label='64 subjects')
plt.hist(aucs_all, bins=bins, alpha=0.6, color='tab:blue', edgecolor='k', label='164 subjects (split by subject)')
plt.xlabel('Test AUC')
plt.ylabel('Number of splits')
plt.title('Test AUC of 200 random splits')
plt.legend()
plt.show()

print(f"64 subjects : mean AUC = {aucs_small.mean():.2f}, standard deviation = {aucs_small.std():.2f}")
print(f"164 subjects: mean AUC = {aucs_all.mean():.2f}, standard deviation = {aucs_all.std():.2f}")

With more subjects, the results are much more stable: the spread is less than half, and the mean AUC is higher. More data gives a better model **and** a more reliable estimate of how good the model is.

### What if we split by row?

For comparison, we run 5-fold cross-validation by **row** (`StratifiedKFold`) and by **subject** (`GroupKFold`).

In [ ]:
by_row = cross_val_score(pipeline, X_all, y_all, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring='roc_auc')
by_subject = cross_val_score(pipeline, X_all, y_all, cv=GroupKFold(5), groups=groups_all, scoring='roc_auc')
print(f"Split by row    : AUC = {by_row.mean():.3f}")
print(f"Split by subject: AUC = {by_subject.mean():.3f}")

Here the split by row gives a slightly higher AUC. The difference is small, because most subjects have only one or two rows, and their recordings are not identical. In datasets with many similar rows per subject, the difference can be very large. You will see an example in Assignment 8.

## 4. Tune the model on the larger dataset

We use the same tuning pipeline as in Section 1, but now with a group-aware test split and `GroupKFold` inside the grid search.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
train_idx, test_idx = next(gss.split(X_all, y_all, groups=groups_all))
X_tr, X_te = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_tr, y_te = y_all.iloc[train_idx], y_all.iloc[test_idx]
g_tr = groups_all.iloc[train_idx]
print("Train subjects:", g_tr.nunique(), "| test subjects:", groups_all.iloc[test_idx].nunique())

grid_all = GridSearchCV(tune_pipeline, param_grid, cv=GroupKFold(5), scoring='roc_auc')
grid_all.fit(X_tr, y_tr, groups=g_tr)

best_all = grid_all.best_estimator_
cv_std = grid_all.cv_results_['std_test_score'][grid_all.best_index_]
print("Best parameters:", grid_all.best_params_)
print(f"Cross-validation AUC (training set): {grid_all.best_score_:.2f} ± {cv_std:.2f}")
print(f"Test AUC: {roc_auc_score(y_te, best_all.predict_proba(X_te)[:, 1]):.2f}")
print("Selected features:", list(X_all.columns[best_all.named_steps['select'].get_support()]))

ConfusionMatrixDisplay.from_estimator(best_all, X_te, y_te, display_labels=['control', 'PD'], cmap='Blues')
plt.title('Tuned model, 164-subject dataset (test set)')
plt.show()

The test AUC (0.79) is lower than the cross-validation AUC (0.91 ± 0.06). This is not a surprise:
- The test set has only 41 subjects, and the histogram above shows that one split can easily change the AUC by 0.05 to 0.10.
- The best cross-validation score is a little optimistic, because we selected it as the **best** of 20 combinations.

Report both numbers, and report the spread. Do not report only the best one.

## Key points

- **With 64 subjects, the result depends on the split.** The same model gave a test AUC from 0.48 to 1.00 with different values of `random_state`. Over 200 splits, the AUC was 0.67 ± 0.15.
- **Never report one lucky split.** Report the mean and the spread over many splits (repeated cross-validation).
- **Do not trust small differences between models** on a small dataset. Our "tuned" model and the first model were not clearly different.
- **More subjects give a better and more stable estimate.** With 164 subjects, the AUC was 0.85 ± 0.06.
- **Split by subject** when subjects have more than one row, and put every step that learns from the data (scaling, feature selection) **inside** the pipeline.
- **Check where the data comes from.** The larger dataset combines three studies with different protocols. A model can learn differences between studies.